In [1]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd

df = pd.read_csv('../../data/csv/sentiment_data_long.csv')


In [2]:
# 독립, 종속변수 분리
X = df['text']
y = df['label']


In [3]:
# 훈련, 테스트 분리 
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [4]:
# 형태소 분리 시 모든 형태소를 포함
# 분리한 단어들을 합침
from konlpy.tag import Okt

def get_preprocessing(sentence):
	okt = Okt()
	result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
	words = [word for word, pos in result]
	return " ".join(words)

# X_train과 X_test의 있는 문장들을 get_preprocessing을 적용
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

In [5]:
# 벡터화 
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode="int",
	output_sequence_length=10
)
vectorize_layer.adapt(X_train.tolist())

In [6]:
import tensorflow as tf

# 모델 설계
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer, 
	layers.Embedding(input_dim=1000, output_dim=64),
	layers.LSTM(32),
	layers.Dense(1, activation='sigmoid')#출력층
])

In [7]:
model.compile(
	optimizer='adam', 
	loss='binary_crossentropy', 
	metrics=['accuracy']
) 

In [8]:
history = model.fit(X_train.values, y_train, epochs=50, verbose=1)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9337 - loss: 0.4782
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0650
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0153
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0084
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0054
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0037
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0027
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0021
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0017
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0014
Epoch 11/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0012
Epoch 12/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - l

In [9]:
# 평가
_, acc = model.evaluate(X_test.values, y_test)
print(f'정확도 : {acc}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 1.2603e-04  
정확도 : 1.0


In [10]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
import numpy as np
sentences = [
	"너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요",
	"지루한 장면이 반복되어서 스토리가 너무 개연성이 없고 결말이 허무하기 짝이 없다"
]
# 학습 데이터가 긍정=> 부정, 부정=>긍정인 데이터들이어서 
# 긍정 문구가 있으면 부정으로 될거라고 예측
# 부정문구가 있으면 긍정으로 될거라고 예측

# 1번 문장은 예측 성공
# 2번 문장은 예측 실패
# => 왜? 학습시킨 모든 데이터가 A => B로 갈 때 다 반전
# => 학습 데이터가 편향

# 머신 러닝/단순한 MLP 방식의 문장 분석
# => 각 단어들이 긍정에 많은지, 부정에 많은지를 계산해서
#    그 결과를 출력
# => 앞에 긍정적인 긴 내요이다가 뒤에 짧은 부정이면 긍정으로 판별
# RNN 방식의 문장 분석
# => 문장의 순서와 단어 사이의 관계를 학습
# => 앞에 긍정적인 긴 내용이다가 뒤에 짧은 부정이면
#    학습 내용에 따라 부정으로 판별할 수 있음
#    => 학습 데이터에 반전된 문장이 없으면 판별 못함
#    => 학습 데이터에 반전된 문장이 있으면 판별함
predictions = model.predict(np.array(sentences,dtype=object))
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step


array([[0.99916816],
       [0.99381036]], dtype=float32)